# Embedding multimodal data for similarity search using 🤗 transformers, 🤗 datasets and FAISS

Embeddings are semantically meaningful compressions of information. They can be used to do similarity search, zero-shot classification or simply train a new model. Use cases for similarity search include searching for similar products in e-commerce, content search in social media and more.
This notebook walks you through using 🤗transformers, 🤗datasets and FAISS to create and index embeddings from a feature extraction model to later use them for similarity search.
Let's install necessary libraries.

In [ ]:
!pip install dill>=0.3.9 dill>=0.3.9
!pip install torchvision 
!pip install -q datasets faiss-gpu transformers sentencepiece

For this tutorial, we will use [CLIP model](https://huggingface.co/openai/clip-vit-base-patch16) to extract the features. CLIP is a revolutionary model that introduced joint training of a text encoder and an image encoder to connect two modalities.

In [ ]:
import torch
from PIL import Image
from transformers import AutoImageProcessor, AutoModel, AutoTokenizer
import faiss
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else "cpu")

model = AutoModel.from_pretrained("openai/clip-vit-base-patch16").to(device)
processor = AutoImageProcessor.from_pretrained("openai/clip-vit-base-patch16")
tokenizer = AutoTokenizer.from_pretrained("openai/clip-vit-base-patch16")

Load the dataset. To keep this notebook light, we will use a small captioning dataset, [jmhessel/newyorker_caption_contest](https://huggingface.co/datasets/jmhessel/newyorker_caption_contest).

In [ ]:
from datasets import load_dataset

ds = load_dataset("jmhessel/newyorker_caption_contest", "explanation")

See an example.

In [ ]:
ds["train"][0]["image"]

In [ ]:
ds["train"][0]["image_description"]

'Two women are looking out a window. There is snow outside, and there is a snowman with human arms.'

We don't have to write any function to embed examples or create an index. 🤗 datasets library's FAISS integration abstracts these processes. We can simply use `map` method of the dataset to create a new column with the embeddings for each example like below. Let's create one for text features on the prompt column.

In [ ]:
dataset = ds["train"]
ds_with_embeddings = dataset.map(lambda example:
                                {'embeddings': model.get_text_features(
                                    **tokenizer([example["image_description"]],
                                                truncation=True, return_tensors="pt")
                                    .to("cuda"))[0].detach().cpu().numpy()})


We can do the same and get the image embeddings.

In [ ]:
ds_with_embeddings = ds_with_embeddings.map(lambda example:
                                          {'image_embeddings': model.get_image_features(
                                              **processor([example["image"]], return_tensors="pt")
                                              .to("cuda"))[0].detach().cpu().numpy()})


Now, we create an index for each column.

In [ ]:
# create FAISS index for text embeddings
ds_with_embeddings.add_faiss_index(column='embeddings')

In [ ]:
# create FAISS index for image embeddings
ds_with_embeddings.add_faiss_index(column='image_embeddings')

## Querying the data with image prompts

Image similarity inference is similar, where you just call `get_image_features`.

In [ ]:
def downscale_images(image):
  width = 200
  ratio = (width / float(image.size[0]))
  height = int((float(image.size[1]) * float(ratio)))
  img = image.resize((width, height), Image.Resampling.LANCZOS)
  return img

In [ ]:
import requests
# image of a beaver
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/beaver.png"
image = Image.open(requests.get(url, stream=True).raw)
display(downscale_images(image))

Search for the similar image.

In [ ]:
img_embedding = model.get_image_features(**processor([image], return_tensors="pt", truncation=True).to("cuda"))[0].detach().cpu().numpy()
scores, retrieved_examples = ds_with_embeddings.get_nearest_examples('image_embeddings', img_embedding, k=1)

Display the most similar image to the beaver image.

In [ ]:
images = [downscale_images(image) for image in retrieved_examples["image"]]
# see the closest text and image
print(retrieved_examples["image_description"])
display(images[0])

## Saving, pushing and loading the embeddings
We can save the dataset with embeddings with `save_faiss_index`.


In [ ]:
ds_with_embeddings.save_faiss_index('embeddings', 'embeddings/embeddings.faiss')

In [ ]:
ds_with_embeddings.save_faiss_index('image_embeddings', 'embeddings/image_embeddings.faiss')

In [ ]:
ds_with_embeddings.drop_index("embeddings")
ds_with_embeddings.drop_index("image_embeddings")

In [ ]:
# save to disk 
ds_with_embeddings.save_to_disk('ds_embeddings')

# save to disk 
dataset.save_to_disk('ds')

In [ ]:
# save to s3
import sagemaker

bucket = "programboy-sagemaker-example"
sess = sagemaker.Session(default_bucket=bucket)

ds_embeddings_uri = sess.upload_data(path='./ds', bucket=sess.default_bucket(), key_prefix='example/ds')
print(ds_embeddings_uri)

embeddings_uri = sess.upload_data(path='./embeddings', bucket=sess.default_bucket(), key_prefix='example/index')
print(embeddings_uri)